# Indonesian License Plate OCR Baseline

Eksperimen ini mengubah gambar plat menjadi string nomor polisi. Dataset berisi gambar, anotasi Pascal VOC XML dengan kelas `plate`, dan sebagian ground truth yang tersimpan pada nama file.

Alur: **gambar -> kandidat crop plat -> preprocessing -> EasyOCR -> normalisasi -> evaluasi**. XML dipakai sebagai ground-truth lokasi/crop, bukan sebagai hasil deteksi model. Untuk foto kendaraan utuh produksi, tahap crop nantinya harus diganti dengan YOLO yang dilatih khusus kelas `plate`.

## 0. Environment
Pilih kernel `Python (.venv) - Smart Parking System`. Dependency sudah tercatat di `requirements.txt`. Jika kernel lama masih aktif setelah instalasi, restart kernel terlebih dahulu.

In [ ]:
# Jalankan hanya jika dependency environment belum dipasang.
# %pip install -r ../../requirements.txt

import csv
import re
import statistics
import xml.etree.ElementTree as ET
from difflib import SequenceMatcher
from pathlib import Path
from time import perf_counter

import cv2
import easyocr
import numpy as np
from IPython.display import Image as DisplayImage, display

print(f'OpenCV: {cv2.__version__}')
print(f'EasyOCR: {easyocr.__version__}')

## 1. Project paths

In [ ]:
def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'apps' / 'vision').is_dir() and (candidate / 'ml').is_dir():
            return candidate
    raise FileNotFoundError('Smart Parking System project root was not found.')


PROJECT_ROOT = find_project_root(Path.cwd())
DATASET_ROOT = PROJECT_ROOT / 'data' / 'raw' / 'indonesian-plate-number'
DATASET_DIR = DATASET_ROOT / 'plate_number_recognition_datasets_v1'
IMAGE_DIR = DATASET_DIR / 'images'
ANNOTATION_DIR = DATASET_DIR / 'annotations'
EXPERIMENT_DIR = PROJECT_ROOT / 'ml' / 'experiments' / 'license_plate_ocr_baseline'
RESULT_CSV = EXPERIMENT_DIR / 'license_plate_ocr_results.csv'

if not IMAGE_DIR.is_dir() or not ANNOTATION_DIR.is_dir():
    raise FileNotFoundError(f'Dataset is incomplete: {DATASET_DIR}')

EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Project: {PROJECT_ROOT}')
print(f'Dataset: {DATASET_DIR}')
print(f'Output:  {EXPERIMENT_DIR}')

## 2. Build image, annotation, and ground-truth records
Nama seperti `351.E 6730 RC-07-19.jpeg` menghasilkan ground truth `E6730RC`. File `IMG-...` tidak mempunyai label teks dan tetap bisa dipakai untuk inspeksi, tetapi tidak masuk metrik OCR.

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}


def normalize_plate_text(value: str) -> str:
    return re.sub(r'[^A-Z0-9]', '', value.upper())


def ground_truth_from_filename(path: Path) -> str | None:
    match = re.match(r'^\d+\.(.+?)-\d{2}-\d{2}$', path.stem)
    return normalize_plate_text(match.group(1)) if match else None


def annotation_path_for(image_path: Path) -> Path:
    split = image_path.parent.name
    return ANNOTATION_DIR / split / f'{image_path.stem}.xml'


records = []
for split in ('train', 'test'):
    for image_path in sorted((IMAGE_DIR / split).iterdir()):
        if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        annotation_path = annotation_path_for(image_path)
        records.append({'split': split, 'image_path': image_path, 'annotation_path': annotation_path, 'ground_truth': ground_truth_from_filename(image_path)})

labeled_count = sum(record['ground_truth'] is not None for record in records)
missing_xml = sum(not record['annotation_path'].is_file() for record in records)
print(f'Total images: {len(records):,}')
print(f'Images with text ground truth: {labeled_count:,}')
print(f'Images without text ground truth: {len(records) - labeled_count:,}')
print(f'Missing XML annotations: {missing_xml:,}')

## 3. Parse and inspect plate bounding boxes
XML dapat memiliki lebih dari satu objek `plate`. Notebook memperluas setiap bounding box sedikit agar karakter tepi tidak terpotong. Bila anotasi tidak tersedia, seluruh gambar menjadi kandidat fallback.

In [ ]:
def read_voc_boxes(annotation_path: Path) -> list[tuple[int, int, int, int]]:
    if not annotation_path.is_file():
        return []
    root = ET.parse(annotation_path).getroot()
    boxes = []
    for obj in root.findall('object'):
        if (obj.findtext('name') or '').strip().lower() != 'plate':
            continue
        bbox = obj.find('bndbox')
        if bbox is not None:
            boxes.append(tuple(int(float(bbox.findtext(key, '0'))) for key in ('xmin', 'ymin', 'xmax', 'ymax')))
    return boxes


def crop_plate_candidates(image: np.ndarray, boxes: list[tuple[int, int, int, int]], margin: float = 0.08) -> list[np.ndarray]:
    height, width = image.shape[:2]
    candidates = []
    for x1, y1, x2, y2 in boxes:
        box_width, box_height = x2 - x1, y2 - y1
        pad_x, pad_y = int(box_width * margin), int(box_height * margin)
        left, top = max(0, x1 - pad_x), max(0, y1 - pad_y)
        right, bottom = min(width, x2 + pad_x), min(height, y2 + pad_y)
        if right > left and bottom > top:
            candidates.append(image[top:bottom, left:right])
    candidates.append(image)  # Most labeled samples are already tightly cropped plates.
    return candidates


sample = next(record for record in records if record['ground_truth'])
sample_image = cv2.imread(str(sample['image_path']))
sample_boxes = read_voc_boxes(sample['annotation_path'])
preview = sample_image.copy()
for x1, y1, x2, y2 in sample_boxes:
    cv2.rectangle(preview, (x1, y1), (x2, y2), (0, 255, 0), 2)
ok, encoded = cv2.imencode('.jpg', preview)
print(f"Ground truth: {sample['ground_truth']} | boxes: {len(sample_boxes)}")
if ok:
    display(DisplayImage(data=encoded.tobytes()))

## 4. Initialize OCR
Pada eksekusi pertama EasyOCR mengunduh model bahasa Inggris. `gpu=False` dipakai karena PyTorch environment ini adalah build CPU.

In [ ]:
OCR_USE_GPU = False
reader = easyocr.Reader(['en'], gpu=OCR_USE_GPU)
print('EasyOCR reader is ready.')

## 5. Preprocessing and OCR helpers
Tiga varian diuji dan proses berhenti lebih cepat ketika hasil berformat plat dengan confidence tinggi ditemukan. Ini menjaga baseline tetap masuk akal di CPU.

In [ ]:
OCR_ALLOWLIST = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'
PLATE_PATTERN = re.compile(r'^[A-Z]{1,3}[0-9]{1,4}[A-Z]{0,3}$')


def preprocess_variants(crop: np.ndarray) -> list[tuple[str, np.ndarray]]:
    height, _ = crop.shape[:2]
    scale = max(1.0, 96 / max(height, 1))
    resized = cv2.resize(crop, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
    _, otsu = cv2.threshold(clahe, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return [('original', resized), ('clahe', clahe), ('otsu', otsu)]


def ocr_variant(image: np.ndarray) -> tuple[str, float]:
    detections = reader.readtext(image, detail=1, paragraph=False, allowlist=OCR_ALLOWLIST)
    if not detections:
        return '', 0.0

    # Group OCR fragments by visual line so the expiry date is not appended to the plate number.
    ordered = sorted(detections, key=lambda item: np.mean([point[1] for point in item[0]]))
    lines = []
    for detection in ordered:
        box = detection[0]
        center_y = float(np.mean([point[1] for point in box]))
        height = max(point[1] for point in box) - min(point[1] for point in box)
        target_line = next((line for line in lines if abs(center_y - line['center_y']) <= 0.6 * max(height, line['height'])), None)
        if target_line is None:
            lines.append({'center_y': center_y, 'height': height, 'items': [detection]})
        else:
            target_line['items'].append(detection)

    candidates = []
    for line in lines:
        items = sorted(line['items'], key=lambda item: min(point[0] for point in item[0]))
        text = normalize_plate_text(''.join(item[1] for item in items))
        confidence = float(np.mean([item[2] for item in items]))
        candidates.append((text, confidence, line['center_y']))

    valid = [candidate for candidate in candidates if PLATE_PATTERN.fullmatch(candidate[0])]
    if valid:
        text, confidence, _ = max(valid, key=lambda candidate: candidate[1])
        return text, confidence

    # The main registration number is normally the upper line on an Indonesian plate.
    text, confidence, _ = min(candidates, key=lambda candidate: candidate[2])
    return text, confidence


def recognize_plate(image: np.ndarray, boxes: list[tuple[int, int, int, int]]) -> dict:
    best = {'prediction': '', 'confidence': 0.0, 'variant': 'none', 'candidate_index': -1}
    for candidate_index, candidate in enumerate(crop_plate_candidates(image, boxes)):
        if candidate.size == 0:
            continue
        for variant_name, variant in preprocess_variants(candidate):
            prediction, confidence = ocr_variant(variant)
            format_bonus = 0.15 if PLATE_PATTERN.fullmatch(prediction) else 0.0
            best_bonus = 0.15 if PLATE_PATTERN.fullmatch(best['prediction']) else 0.0
            if prediction and confidence + format_bonus > best['confidence'] + best_bonus:
                best = {'prediction': prediction, 'confidence': confidence, 'variant': variant_name, 'candidate_index': candidate_index}
            if PLATE_PATTERN.fullmatch(prediction) and confidence >= 0.85:
                return best
    return best

## 6. Run baseline
Mulai dari split `test`. Ubah `MAX_IMAGES = None` setelah alur terverifikasi untuk memproses seluruh split.

In [ ]:
EVALUATION_SPLIT = 'test'
MAX_IMAGES = 30  # None untuk seluruh split
ONLY_LABELED_IMAGES = True

evaluation_records = [record for record in records if record['split'] == EVALUATION_SPLIT and (record['ground_truth'] is not None or not ONLY_LABELED_IMAGES)]
if MAX_IMAGES is not None:
    evaluation_records = evaluation_records[:MAX_IMAGES]

results = []
for index, record in enumerate(evaluation_records, start=1):
    image = cv2.imread(str(record['image_path']))
    if image is None:
        print(f"Unreadable image: {record['image_path'].name}")
        continue
    boxes = read_voc_boxes(record['annotation_path'])
    started_at = perf_counter()
    recognition = recognize_plate(image, boxes)
    elapsed_ms = (perf_counter() - started_at) * 1000
    ground_truth, prediction = record['ground_truth'], recognition['prediction']
    exact_match = ground_truth is not None and prediction == ground_truth
    similarity = SequenceMatcher(None, ground_truth or '', prediction).ratio() if ground_truth else None
    results.append({
        'split': record['split'], 'filename': record['image_path'].name, 'ground_truth': ground_truth or '',
        'prediction': prediction, 'confidence': round(recognition['confidence'], 4), 'exact_match': exact_match,
        'similarity': round(similarity, 4) if similarity is not None else '', 'variant': recognition['variant'],
        'candidate_index': recognition['candidate_index'], 'box_count': len(boxes), 'ocr_ms': round(elapsed_ms, 2),
    })
    print(f"[{index:02d}/{len(evaluation_records):02d}] GT={ground_truth or '-':10s} OCR={prediction or '-':10s} conf={recognition['confidence']:.2f}")

## 7. Metrics and CSV export

In [ ]:
if not results:
    raise RuntimeError('No OCR results were produced.')

with RESULT_CSV.open('w', newline='', encoding='utf-8-sig') as handle:
    writer = csv.DictWriter(handle, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)

labeled_results = [row for row in results if row['ground_truth']]
exact_matches = sum(row['exact_match'] for row in labeled_results)
similarities = [float(row['similarity']) for row in labeled_results]
latencies = [float(row['ocr_ms']) for row in results]
print(f'Processed: {len(results):,}')
print(f'Exact-match accuracy: {exact_matches / len(labeled_results):.2%}' if labeled_results else 'Exact-match accuracy: N/A')
print(f'Mean character similarity: {statistics.mean(similarities):.2%}' if similarities else 'Mean character similarity: N/A')
print(f'Median OCR latency: {statistics.median(latencies):.1f} ms/image')
print(f'CSV: {RESULT_CSV}')

## 8. Review errors visually

In [ ]:
error_rows = sorted((row for row in results if row['ground_truth'] and not row['exact_match']), key=lambda row: float(row['similarity']))[:8]
for row in error_rows:
    image = cv2.imread(str(IMAGE_DIR / row['split'] / row['filename']))
    if image is None:
        continue
    ok, encoded = cv2.imencode('.jpg', image)
    print(f"GT: {row['ground_truth']} | OCR: {row['prediction'] or '-'} | confidence: {row['confidence']} | similarity: {row['similarity']}")
    if ok:
        display(DisplayImage(data=encoded.tobytes(), width=600))

## 9. Interpretation and next step

- **Recognition baseline:** exact-match accuracy adalah metrik utama; similarity membantu melihat kesalahan satu-dua karakter.
- **Detection baseline:** XML adalah ground truth, bukan prediksi. Dataset didominasi crop plat dan beberapa gambar memiliki anotasi yang perlu diaudit, sehingga belum cukup representatif untuk detector kamera parkir.
- **Next experiment:** kumpulkan/label foto kendaraan utuh dari sudut kamera parkir, konversi bounding box ke format YOLO, lalu fine-tune detector kelas `plate`. Output crop detector tersebut masuk ke fungsi OCR notebook ini.
- Jangan membaca ground truth dari nama file sebagai prediksi; nama file hanya dipakai untuk evaluasi.

## 10. Upload your own plate image

Upload satu gambar crop/close-up plat, isi GT secara opsional, lalu klik **Run OCR**. File hanya diproses dari memory dan tidak disimpan ke dataset. Foto kendaraan utuh belum didukung sampai detector YOLO kelas `plate` tersedia.

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output

upload_widget = widgets.FileUpload(accept='image/*', multiple=False, description='Choose image')
ground_truth_widget = widgets.Text(description='GT (optional)', placeholder='Contoh: B1234XYZ')
run_button = widgets.Button(description='Run OCR', button_style='primary', icon='search')
upload_output = widgets.Output()


def uploaded_file_payload(value):
    if isinstance(value, dict):
        return next(iter(value.values()), None)
    return value[0] if value else None


def run_uploaded_image_ocr(_):
    with upload_output:
        clear_output(wait=True)
        payload = uploaded_file_payload(upload_widget.value)
        if payload is None:
            print('Pilih satu file gambar terlebih dahulu.')
            return

        image_bytes = bytes(payload['content'])
        image = cv2.imdecode(np.frombuffer(image_bytes, dtype=np.uint8), cv2.IMREAD_COLOR)
        if image is None:
            print('File tidak dapat dibaca sebagai gambar.')
            return

        started_at = perf_counter()
        recognition = recognize_plate(image, boxes=[])
        elapsed_ms = (perf_counter() - started_at) * 1000
        prediction = recognition['prediction']
        optional_gt = normalize_plate_text(ground_truth_widget.value)

        ok, encoded = cv2.imencode('.jpg', image)
        if ok:
            display(DisplayImage(data=encoded.tobytes(), width=700))
        print(f"OCR: {prediction or 'Tidak terbaca'}")
        print(f"Confidence: {recognition['confidence']:.2%}")
        print(f"Preprocessing: {recognition['variant']}")
        print(f"Waktu proses: {elapsed_ms:.1f} ms")
        if optional_gt:
            similarity = SequenceMatcher(None, optional_gt, prediction).ratio()
            print(f'GT: {optional_gt}')
            print(f'Exact match: {prediction == optional_gt}')
            print(f'Character similarity: {similarity:.2%}')


run_button.on_click(run_uploaded_image_ocr)
display(widgets.VBox([upload_widget, ground_truth_widget, run_button, upload_output]))